In [3]:
import yfinance as YF

def fetch_yf_data(ticker: str):
    df = YF.Ticker(ticker).history(period="max")
    df.index = df.index.tz_localize(None)
    return df

aapl = fetch_yf_data("AAPL")

In [4]:
aapl[aapl.index>="2025-01-01"].Close

Date
2025-01-02    243.263199
2025-01-03    242.774368
2025-01-06    244.410416
2025-01-07    241.627136
2025-01-08    242.115952
                 ...    
2025-06-02    201.699997
2025-06-03    203.270004
2025-06-04    202.820007
2025-06-05    200.630005
2025-06-06    203.919998
Name: Close, Length: 107, dtype: float64

In [28]:
import pandas as pd
import numpy as np
class TimeSeries:
    """time series object to store the historical and expected performance of an asset"""    
    def __init__(self, ticker: str, data: pd.DataFrame, quantity: float = 1, expected_performance: float = 0.06):
        """Intialize a new time series and evaluate automatically the expected performance

        Args:
            ticker (str): YFinance ticker
            data (pd.DataFrame): dataframe with the historical data
            quantity (float, optional): number of shares owned. Defaults to 1.
            _expected_performance (float, optional): expected performance of the asset. Defaults to 0.06.
        """        
        self.ticker = ticker
        self.start_date = data.index[0]
        self._data = data 
        self._quantity = quantity
        self._expected_performance = expected_performance

        self.calculate_asset_performance()
        self.calculate_expected_value()
    
    @property
    def data(self):
        return self._data*self._quantity
        
    def calculate_asset_performance(self):
        self._data["ActualPerformance"] = (self._data["Close"] / self._data["Close"].iloc[0]) - 1
        
    def calculate_expected_value(self, expected_performance: float = None):
        if expected_performance is None:
            expected_performance = self._expected_performance
        data = self._data.copy()
        data.index = pd.to_datetime(data.index)
        t0 = data.index[0]

        days_elapsed = (data.index - t0).days
        self._data["ExpectedDevFactor"] = (1 + expected_performance) ** (days_elapsed / 365)
        self._data["ExpectedValue"] = self._data["Close"].iloc[0] * self._data["ExpectedDevFactor"]
        pass
    

In [29]:
apple_ts = TimeSeries("AAPL", data = aapl[aapl.index >= "2025-01-01"][["Close", "Dividends", "Stock Splits"]], quantity = 1, expected_performance = 0.06)
apple_ts.data

,Close,Dividends,Stock Splits,ActualPerformance,ExpectedDevFactor,ExpectedValue
Date,,,,,,
2025-01-02,243.263199,0.0,0.0,0.000000,1.000000,243.263199
2025-01-03,242.774368,0.0,0.0,-0.002009,1.000160,243.302037
2025-01-06,244.410416,0.0,0.0,0.004716,1.000639,243.418587
2025-01-07,241.627136,0.0,0.0,-0.006725,1.000799,243.457450
2025-01-08,242.115952,0.0,0.0,-0.004716,1.000958,243.496319
...,...,...,...,...,...,...
2025-06-02,201.699997,0.0,0.0,-0.170857,1.024399,249.198495
2025-06-03,203.270004,0.0,0.0,-0.164403,1.024562,249.238280
2025-06-04,202.820007,0.0,0.0,-0.166253,1.024726,249.278072


Date
2025-01-02    243.263199
2025-01-03    242.774368
2025-01-06    244.410416
2025-01-07    241.627136
2025-01-08    242.115952
                 ...    
2025-06-02    201.699997
2025-06-03    203.270004
2025-06-04    202.820007
2025-06-05    200.630005
2025-06-06    203.919998
Name: Close, Length: 107, dtype: float64